In [0]:

# ============================================================
# 08_HIERARCHY_VALIDATION
# CELL 1: SOURCE TABLE LOADING AND INITIAL VALIDATION
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. DEFINE TABLE NAMES
# ------------------------------------------------------------

HIERARCHY_TABLE = (
    "workspace.firmable_v2_silver.employee_hierarchy"
)

EMPLOYEE_TABLE = (
    "workspace.firmable_v2_silver.employees_clean"
)

ASSIGNMENT_TABLE = (
    "workspace.firmable_v2_silver.manager_assignments"
)

VALIDATION_TABLE = (
    "workspace.firmable_v2_silver.hierarchy_validation_summary"
)


# ------------------------------------------------------------
# 2. LOAD SOURCE TABLES
# ------------------------------------------------------------

hierarchy = spark.table(HIERARCHY_TABLE)

employees = spark.table(EMPLOYEE_TABLE)

assignments = spark.table(ASSIGNMENT_TABLE)


# ------------------------------------------------------------
# 3. VERIFY REQUIRED COLUMNS
# ------------------------------------------------------------

required_hierarchy_columns = {
    "company_id",
    "employee_id",
    "manager_id",
    "root_employee_id",
    "hierarchy_depth",
    "is_root",
    "root_type",
    "inferred_seniority_level"
}

required_employee_columns = {
    "company_id",
    "employee_id",
    "inferred_seniority_level"
}

required_assignment_columns = {
    "company_id",
    "employee_id",
    "candidate_manager_id"
}

missing_hierarchy_columns = (
    required_hierarchy_columns - set(hierarchy.columns)
)

missing_employee_columns = (
    required_employee_columns - set(employees.columns)
)

missing_assignment_columns = (
    required_assignment_columns - set(assignments.columns)
)

assert not missing_hierarchy_columns, (
    f"Missing hierarchy columns: {sorted(missing_hierarchy_columns)}"
)

assert not missing_employee_columns, (
    f"Missing employee columns: {sorted(missing_employee_columns)}"
)

assert not missing_assignment_columns, (
    f"Missing assignment columns: {sorted(missing_assignment_columns)}"
)


# ------------------------------------------------------------
# 4. CALCULATE SOURCE RECORD COUNTS
# ------------------------------------------------------------

hierarchy_count = hierarchy.count()

employee_count = employees.count()

assignment_count = assignments.count()


# ------------------------------------------------------------
# 5. DISPLAY SOURCE INFORMATION
# ------------------------------------------------------------

print("=" * 60)
print("HIERARCHY VALIDATION - SOURCE TABLE LOADING")
print("=" * 60)

print("\nSOURCE TABLES")

print("Hierarchy table:", HIERARCHY_TABLE)
print("Employee table:", EMPLOYEE_TABLE)
print("Assignment table:", ASSIGNMENT_TABLE)

print("\nSOURCE RECORD COUNTS")

print("Hierarchy records:", hierarchy_count)
print("Employee records:", employee_count)
print("Assignment records:", assignment_count)

print("\nSCHEMA VALIDATION")

print("Hierarchy required columns: PASSED")
print("Employee required columns: PASSED")
print("Assignment required columns: PASSED")


# ------------------------------------------------------------
# 6. BASIC SOURCE COUNT CONSISTENCY
# ------------------------------------------------------------

assert hierarchy_count == employee_count, (
    "Hierarchy and employee record counts do not match."
)

assert assignment_count == employee_count, (
    "Assignment and employee record counts do not match."
)

print("\nSOURCE COUNT VALIDATION: PASSED")

print("\nAll source tables loaded successfully.")
print("Cell 1 completed successfully.")


In [0]:

# ============================================================
# CELL 2: BASIC COUNTS AND UNIQUENESS
# ============================================================

# 1. Total hierarchy records
total_employees = hierarchy.count()

# 2. Duplicate employee records in hierarchy
duplicate_employees = (
    hierarchy
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 3. Employees missing from hierarchy
missing_employee_records = (
    employees
    .select("company_id", "employee_id")
    .dropDuplicates()
    .join(
        hierarchy.select("company_id", "employee_id"),
        ["company_id", "employee_id"],
        "left_anti"
    )
    .count()
)

# 4. Extra employees present in hierarchy
extra_employee_records = (
    hierarchy
    .select("company_id", "employee_id")
    .dropDuplicates()
    .join(
        employees
        .select("company_id", "employee_id")
        .dropDuplicates(),
        ["company_id", "employee_id"],
        "left_anti"
    )
    .count()
)

# 5. Display results
print("=" * 60)
print("BASIC COUNTS AND UNIQUENESS VALIDATION")
print("=" * 60)

print("Total hierarchy employees:", total_employees)
print("Duplicate employee records:", duplicate_employees)
print("Missing employee records:", missing_employee_records)
print("Extra employee records:", extra_employee_records)

# 6. Validation
assert duplicate_employees == 0, "Duplicate employees found."
assert missing_employee_records == 0, "Employees missing from hierarchy."
assert extra_employee_records == 0, "Extra employees found in hierarchy."

print("\nCELL 2 VALIDATION PASSED")


In [0]:

# ============================================================
# CELL 3: MANAGER REFERENCE VALIDATION
# ============================================================

# Build a lookup of all employees who can be referenced as managers.
manager_lookup = hierarchy.select(
    "company_id",
    F.col("employee_id").alias("checked_manager_id"),
    F.col("root_employee_id").alias("manager_root_id"),
    F.col("hierarchy_depth").alias("manager_depth"),
    F.col("inferred_seniority_level").alias("manager_level")
)

# Select employees who have an assigned manager.
children = hierarchy.filter(
    F.col("manager_id").isNotNull()
)

# Match each employee to their assigned manager.
manager_check = children.alias("child").join(
    manager_lookup.alias("manager"),
    (
        F.col("child.company_id") == F.col("manager.company_id")
    ) & (
        F.col("child.manager_id") == F.col("manager.checked_manager_id")
    ),
    "left"
)

# Count references that do not resolve to an employee.
invalid_manager_references = manager_check.filter(
    F.col("manager.checked_manager_id").isNull()
).count()

# Display results.
print("=" * 60)
print("MANAGER REFERENCE VALIDATION")
print("=" * 60)

print("Assigned employees:", children.count())
print("Invalid manager references:", invalid_manager_references)

# Validate.
assert invalid_manager_references == 0, (
    f"Found {invalid_manager_references} invalid manager references."
)

print("\nCELL 3 VALIDATION PASSED")


In [0]:

# ============================================================
# CELL 4: SENIORITY VALIDATION
# Lower seniority number means more senior.
# A manager must have a lower seniority level than the employee.
# ============================================================

invalid_seniority = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("manager.manager_level") >=
        F.col("child.inferred_seniority_level")
    )
).count()

# Also identify missing seniority values on either side.
missing_seniority_records = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("manager.manager_level").isNull() |
        F.col("child.inferred_seniority_level").isNull()
    )
).count()

print("=" * 60)
print("SENIORITY VALIDATION")
print("=" * 60)

print("Invalid seniority relationships:", invalid_seniority)
print("Relationships with missing seniority:", missing_seniority_records)

assert invalid_seniority == 0, (
    f"Found {invalid_seniority} invalid seniority relationships."
)

assert missing_seniority_records == 0, (
    f"Found {missing_seniority_records} relationships with missing seniority."
)

print("\nCELL 4 VALIDATION PASSED")


In [0]:

# ============================================================
# CELL 5: HIERARCHY DEPTH VALIDATION
# Every employee must be exactly one level below their manager.
# ============================================================

invalid_depth = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("child.hierarchy_depth") !=
        F.col("manager.manager_depth") + 1
    )
).count()

missing_depth_records = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("child.hierarchy_depth").isNull() |
        F.col("manager.manager_depth").isNull()
    )
).count()

print("=" * 60)
print("HIERARCHY DEPTH VALIDATION")
print("=" * 60)

print("Invalid hierarchy depth relationships:", invalid_depth)
print("Relationships with missing depth:", missing_depth_records)

assert invalid_depth == 0, (
    f"Found {invalid_depth} invalid hierarchy depth relationships."
)

assert missing_depth_records == 0, (
    f"Found {missing_depth_records} relationships with missing depth."
)

print("\nCELL 5 VALIDATION PASSED")


In [0]:

# ============================================================
# CELL 6: ROOT CONSISTENCY VALIDATION
# ============================================================

# 1. Validate root records and non-root records.
invalid_root_records = hierarchy.filter(
    F.col("root_employee_id").isNull() |
    F.col("hierarchy_depth").isNull() |
    (
        F.col("is_root") &
        (
            (F.col("root_employee_id") != F.col("employee_id")) |
            (F.col("hierarchy_depth") != 0)
        )
    ) |
    (
        (~F.col("is_root")) &
        (F.col("hierarchy_depth") <= 0)
    )
).count()


# 2. Verify that every root_employee_id references an actual root.
root_lookup = hierarchy.filter(
    F.col("is_root")
).select(
    "company_id",
    F.col("employee_id").alias("checked_root_id")
)

invalid_root_references = (
    hierarchy
    .join(
        root_lookup,
        (
            hierarchy.company_id == root_lookup.company_id
        ) &
        (
            hierarchy.root_employee_id == root_lookup.checked_root_id
        ),
        "left"
    )
    .filter(F.col("checked_root_id").isNull())
    .count()
)


# 3. Verify that employees and their managers share the same root.
invalid_component_links = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("child.root_employee_id") !=
        F.col("manager.manager_root_id")
    )
).count()


# 4. Display results.
print("=" * 60)
print("ROOT CONSISTENCY VALIDATION")
print("=" * 60)

print("Invalid root records:", invalid_root_records)
print("Invalid root references:", invalid_root_references)
print("Invalid manager-component links:", invalid_component_links)


# 5. Validation assertions.
assert invalid_root_records == 0, (
    f"Found {invalid_root_records} invalid root records."
)

assert invalid_root_references == 0, (
    f"Found {invalid_root_references} invalid root references."
)

assert invalid_component_links == 0, (
    f"Found {invalid_component_links} invalid component links."
)

print("\nCELL 6 VALIDATION PASSED")


In [0]:

# ============================================================
# CELL 6: COMPLETE REMAINING HIERARCHY VALIDATION
# Root Consistency, Assignment Reconciliation, Cycle Safety,
# Integrity Checks and Final Validation Report
# ============================================================

from pyspark.sql import functions as F

# ============================================================
# 1. ROOT CONSISTENCY VALIDATION
# ============================================================

invalid_root_records = hierarchy.filter(
    F.col("root_employee_id").isNull() |
    F.col("hierarchy_depth").isNull() |
    F.col("is_root").isNull() |
    (
        F.col("is_root") &
        (
            (F.col("root_employee_id") != F.col("employee_id")) |
            (F.col("hierarchy_depth") != 0)
        )
    ) |
    (
        (~F.col("is_root")) &
        (F.col("hierarchy_depth") <= 0)
    )
).count()


# ============================================================
# 2. ROOT REFERENCE VALIDATION
# Every root_employee_id must reference an actual root.
# ============================================================

root_lookup = hierarchy.filter(
    F.col("is_root") == True
).select(
    "company_id",
    F.col("employee_id").alias("checked_root_id")
)

invalid_root_references = (
    hierarchy.alias("h")
    .join(
        root_lookup.alias("r"),
        (
            F.col("h.company_id") == F.col("r.company_id")
        ) &
        (
            F.col("h.root_employee_id") == F.col("r.checked_root_id")
        ),
        "left"
    )
    .filter(F.col("r.checked_root_id").isNull())
    .count()
)


# ============================================================
# 3. MANAGER-COMPONENT CONSISTENCY
# Employees and their managers must share the same root.
# ============================================================

invalid_component_links = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("child.root_employee_id") !=
        F.col("manager.manager_root_id")
    )
).count()


# ============================================================
# 4. SELF-RELATIONSHIP VALIDATION
# ============================================================

self_relationships = hierarchy.filter(
    F.col("employee_id") == F.col("manager_id")
).count()


# ============================================================
# 5. CYCLE SAFETY VALIDATION
#
# Strictly decreasing seniority along every manager-to-child
# edge prevents a directed cycle.
# Seniority violations were already checked in Cell 4.
# ============================================================

cycle_safety_failures = invalid_seniority


# ============================================================
# 6. ASSIGNMENT RECONCILIATION
# Compare actual manager IDs, not just total assignment counts.
# ============================================================

hierarchy_assignments = hierarchy.filter(
    F.col("manager_id").isNotNull()
).select(
    "company_id",
    "employee_id",
    "manager_id"
)

source_assignments = assignments.filter(
    F.col("candidate_manager_id").isNotNull()
).select(
    "company_id",
    "employee_id",
    F.col("candidate_manager_id").alias("source_manager_id")
)

assigned_in_hierarchy = hierarchy_assignments.count()
assigned_in_source = source_assignments.count()

assignment_count_difference = abs(
    assigned_in_hierarchy - assigned_in_source
)

# Check duplicate assignment keys in source and hierarchy.
duplicate_source_assignments = (
    source_assignments
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_hierarchy_assignments = (
    hierarchy_assignments
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Compare manager IDs for every assigned employee.
assignment_comparison = hierarchy_assignments.alias("h").join(
    source_assignments.alias("s"),
    (
        F.col("h.company_id") == F.col("s.company_id")
    ) &
    (
        F.col("h.employee_id") == F.col("s.employee_id")
    ),
    "full_outer"
)

assignment_mapping_mismatches = assignment_comparison.filter(
    F.col("h.employee_id").isNull() |
    F.col("s.employee_id").isNull() |
    (F.col("h.manager_id") != F.col("s.source_manager_id"))
).count()


# ============================================================
# 7. FINAL ROOT AND DEPTH STATISTICS
# ============================================================

total_employees = hierarchy.count()

root_count = hierarchy.filter(
    F.col("is_root") == True
).count()

ceo_roots = hierarchy.filter(
    F.col("root_type") == "CEO_ROOT"
).count()

unassigned_roots = hierarchy.filter(
    F.col("root_type") == "UNASSIGNED_ROOT"
).count()

maximum_depth = hierarchy.agg(
    F.max("hierarchy_depth")
).first()[0]

unreachable_employees = hierarchy.filter(
    F.col("root_employee_id").isNull()
).count()


# ============================================================
# 8. BUILD COMPLETE VALIDATION REPORT
# ============================================================

checks = [
    ("duplicate_employee_records", duplicate_employees),
    ("missing_employee_records", missing_employee_records),
    ("extra_employee_records", extra_employee_records),
    ("invalid_manager_references", invalid_manager_references),
    ("invalid_seniority_relationships", invalid_seniority),
    ("missing_seniority_records", missing_seniority_records),
    ("invalid_hierarchy_depth", invalid_depth),
    ("missing_hierarchy_depth", missing_depth_records),
    ("invalid_root_records", invalid_root_records),
    ("invalid_root_references", invalid_root_references),
    ("invalid_component_links", invalid_component_links),
    ("self_relationships", self_relationships),
    ("cycle_safety_failures", cycle_safety_failures),
    ("assignment_count_difference", assignment_count_difference),
    ("assignment_mapping_mismatches", assignment_mapping_mismatches),
    ("duplicate_source_assignments", duplicate_source_assignments),
    ("duplicate_hierarchy_assignments", duplicate_hierarchy_assignments),
    ("unreachable_employees", unreachable_employees)
]

validation_rows = [
    (name, int(value), value == 0)
    for name, value in checks
]

validation_report = spark.createDataFrame(
    validation_rows,
    ["validation_check", "failure_count", "passed"]
)


# ============================================================
# 9. PUBLISH VALIDATION REPORT
# ============================================================

VALIDATION_TABLE = (
    "workspace.firmable_v2_silver.hierarchy_validation_summary"
)

(
    validation_report.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALIDATION_TABLE)
)


# ============================================================
# 10. FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FIRMABLE - COMPLETE HIERARCHY VALIDATION REPORT")
print("=" * 65)

validation_report.orderBy(
    "validation_check"
).show(100, truncate=False)

print("\n" + "=" * 65)
print("HIERARCHY SUMMARY")
print("=" * 65)

print("Total employees:", total_employees)
print("Assigned employees:", assigned_in_hierarchy)
print("Unassigned employees:", total_employees - assigned_in_hierarchy)
print("Component roots:", root_count)
print("CEO roots:", ceo_roots)
print("Unassigned roots:", unassigned_roots)
print("Maximum hierarchy depth:", maximum_depth)

print("\nASSIGNMENT RECONCILIATION")
print("Assignments in source:", assigned_in_source)
print("Assignments in hierarchy:", assigned_in_hierarchy)
print("Assignment count difference:", assignment_count_difference)
print("Assignment mapping mismatches:", assignment_mapping_mismatches)

print("\nROOT DISTRIBUTION")

hierarchy.filter(
    F.col("is_root") == True
).groupBy(
    "root_type"
).count().orderBy(
    "root_type"
).show(truncate=False)

print("\nHIERARCHY DEPTH DISTRIBUTION")

hierarchy.groupBy(
    "hierarchy_depth"
).count().orderBy(
    "hierarchy_depth"
).show(truncate=False)


# ============================================================
# 11. FINAL PASS / FAIL
# ============================================================

failed_checks = sum(
    1 for _, value in checks if value != 0
)

print("\n" + "=" * 65)

if failed_checks == 0:
    print("RESULT: ALL HIERARCHY VALIDATION CHECKS PASSED")
else:
    print("RESULT: HIERARCHY VALIDATION FAILED")
    print("Number of failed checks:", failed_checks)

assert failed_checks == 0, (
    f"Hierarchy validation failed: {failed_checks} checks have errors."
)

print("Validation report published:", VALIDATION_TABLE)
print("Notebook 08 validation completed successfully.")
